## Solving CartPole with DeepQNetworks


In [4]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


TODO: I plan to write a tutorial on going through how DQN's work, and how it compares to other algorithms like policy gradient methods, and actor critic methods. Deep Q networks solved the stability issue by introducing target networks and replay buffers both of which have interesting extensions that can be extended into the samsara rl library due to its use of the composition pattern. 

Deep Q networks first learn the true values of states adjacent to terminal states. These are points the target network which is initially noise is zero'd out (no future reward) and only the R signal computes to the temporal difference. 

In future iterations, the bootstrap values from the adjacent states propogate to the rest of the state space but require careful tuning, gradient clipping to combat noisy bootstrap estimates. Batch learning  helps reduce variance and move towards stable learning.

Until I get to the full tutorial of how deep q nets work this full Gym example below shows how to use the Deep Q Network in the library. It is capable of running any gym environment, here I used cart pole

In [7]:
import os

import gymnasium as gym
import structlog
from cart_pole_logger import CartPoleLogger

# from samsara_rl.mdp.terminal_penalty_wrapper import TerminalPenaltyWrapper
from samsara_rl.control.function_approximation.batch.deep_q_network.q_network import QNetwork
from samsara_rl.control.function_approximation.functions.neural_networks.fully_connected import (
    FullyConnected,
)
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

In [8]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/double_dqn"
ALPHAS = [0.0001, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [ ]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single configuration.

    Creates a LinearFunction and QLearningGradient agent, trains for
    MAX_EPISODES episodes, saves the learned weights, and runs greedy
    evaluation.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the Q-learning update.
    """
    model_name = "deep_q_network"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    fc = FullyConnected(4, 32, 2, preprocess=None)

    agent = QNetwork(mdp=env, policy=None, gamma=gamma, q=fc, alpha=alpha, log_dir=run_dir, experiment_name=run_name)

    agent.register(CartPoleLogger())

    agent.evaluate(max_iter=MAX_EPISODES)
    #     save_model(agent, run_dir)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [10]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [ ]:
main()

2026-09-01 16:27:38 [info     ] starting_run                   run='deep_q_network_alpha=0.0001_gamma=0.99'


/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
